In [1]:
"""
PINN for the inverse Navier-Stokes problem, cylinder wake Re = 500.

Same network, loss, data and training schedule as CylinderRe500-trial1,
but the run can be stopped at any time and continued later:

  * every Adam / L-BFGS-B iteration (loss, lambda_1, lambda_2) is stored
  * a checkpoint is written regularly and when the run is stopped
    (Ctrl+C, `kill`, closing the terminal under nohup/tmux)
  * starting the script again continues from the last checkpoint
  * all printed lines are also appended to  <CKPT_DIR>/train_log.txt

Run it from a terminal, NOT from Jupyter:

    nohup python -u pinn_re500_resumable.py > console.txt 2>&1 &
    tail -f checkpoints_re500/train_log.txt          # watch progress

Show progress / loss plot at any time (also while it is running):

    python pinn_re500_resumable.py --status

Start completely fresh: delete the checkpoint folder.
"""

import os
import sys
import time
import signal
import argparse

import tensorflow as tf
import scipy.io
from scipy.interpolate import griddata
import numpy as np
from scipy.optimize import minimize
import matplotlib
matplotlib.use("Agg")                    # no display needed (works over nohup/ssh)
import matplotlib.pyplot as plt

np.random.seed(1234)
tf.random.set_seed(1234)

# ----------------------------------------------------------------------
# Settings
# ----------------------------------------------------------------------
DATA_FILE = "cylinder_Re500_mid.mat"
CKPT_DIR = "checkpoints_re500"
LAMBDA_2_TRUE = 0.002               # nu = 1/Re

ADAM_SAVE_EVERY = 1000              # checkpoint every N Adam iterations
LBFGS_SAVE_EVERY = 50               # checkpoint every N L-BFGS-B iterations
ADAM_PRINT_EVERY = 100              # print every N Adam iterations (all are stored)

LOG_FILE = os.path.join(CKPT_DIR, "train_log.txt")


def log(msg):
    """Print and append to the log file (survives browser/terminal crashes)."""
    print(msg, flush=True)
    with open(LOG_FILE, "a") as f:
        f.write(msg + "\n")


def _atomic_savez(path, **arrays):
    """Write .npz to a temp file first, then rename -> never a half-written checkpoint."""
    tmp = path + ".tmp.npz"
    np.savez(tmp, **arrays)
    if os.path.exists(path):
        os.replace(path, path + ".bak")      # keep previous checkpoint as backup
    os.replace(tmp, path)


def _sigterm_handler(signum, frame):
    # `kill <pid>` -> behave like Ctrl+C so a checkpoint is written first;
    # further signals are ignored so the checkpoint is not interrupted
    signal.signal(signal.SIGTERM, signal.SIG_IGN)
    signal.signal(signal.SIGINT, signal.SIG_IGN)
    raise KeyboardInterrupt


signal.signal(signal.SIGTERM, _sigterm_handler)


class PhysicsInformedNN:
    def __init__(self, x, y, t, u, v, layers, learning_rate=1e-3, dtype=tf.float32,
                 run_name="clean"):
        self.dtype = dtype
        self.layers = layers
        self.run_name = run_name
        self.ckpt_file = os.path.join(CKPT_DIR, f"state_{run_name}.npz")

        X = np.concatenate([x, y, t], axis=1).astype(np.float64)

        self.lb = tf.constant(X.min(0, keepdims=True), dtype=self.dtype)
        self.ub = tf.constant(X.max(0, keepdims=True), dtype=self.dtype)

        self.X = tf.convert_to_tensor(X, dtype=self.dtype)
        self.x = tf.convert_to_tensor(X[:, 0:1], dtype=self.dtype)
        self.y = tf.convert_to_tensor(X[:, 1:2], dtype=self.dtype)
        self.t = tf.convert_to_tensor(X[:, 2:3], dtype=self.dtype)

        self.u = tf.convert_to_tensor(u, dtype=self.dtype)
        self.v = tf.convert_to_tensor(v, dtype=self.dtype)

        # ---------------------------
        # Network
        # ---------------------------
        self.net = self._build_network(layers)

        # Build once so trainable variables exist immediately
        _ = self.net(tf.zeros((1, layers[0]), dtype=self.dtype))

        # Trainable PDE parameters
        self.lambda_1 = tf.Variable(0.0, dtype=self.dtype, trainable=True, name="lambda_1")
        self.lambda_2 = tf.Variable(0.0, dtype=self.dtype, trainable=True, name="lambda_2")

        # Optimizer (built now so its state can be restored from a checkpoint)
        self.optimizer_adam = tf.keras.optimizers.Adam(learning_rate=learning_rate)
        self.optimizer_adam.build(self.train_vars)

        # Progress counters and full history (every iteration)
        self.stage = "adam"          # "adam" -> "lbfgs" -> "done"
        self.adam_it = 0
        self.lbfgs_it = 0
        self.lbfgs_nfev = 0
        self.hist = {k: [] for k in
                     ["adam_loss", "adam_l1", "adam_l2",
                      "lbfgs_loss", "lbfgs_l1", "lbfgs_l2"]}

        self.load_checkpoint()

    # ------------------------------------------------------------------
    # Checkpointing
    # ------------------------------------------------------------------
    def _opt_vars(self):
        v = self.optimizer_adam.variables
        return v() if callable(v) else v

    def save_checkpoint(self, weights=None):
        """weights: flat vector to store (default: current weights)."""
        if weights is None:
            weights = self._pack_weights()
        arrays = dict(
            stage=np.array(self.stage),
            adam_it=np.array(self.adam_it),
            lbfgs_it=np.array(self.lbfgs_it),
            lbfgs_nfev=np.array(self.lbfgs_nfev),
            weights=weights,
            n_opt=np.array(len(self._opt_vars())),
        )
        for i, var in enumerate(self._opt_vars()):
            arrays[f"opt_{i}"] = np.array(var.numpy())
        for k, val in self.hist.items():
            arrays[k] = np.array(val, dtype=np.float64)
        _atomic_savez(self.ckpt_file, **arrays)

    def load_checkpoint(self):
        if not os.path.exists(self.ckpt_file):
            return
        d = np.load(self.ckpt_file)
        self.stage = str(d["stage"])
        self.adam_it = int(d["adam_it"])
        self.lbfgs_it = int(d["lbfgs_it"])
        self.lbfgs_nfev = int(d["lbfgs_nfev"])
        self._unpack_weights(d["weights"])
        opt_vars = self._opt_vars()
        if int(d["n_opt"]) == len(opt_vars):
            for i, var in enumerate(opt_vars):
                var.assign(d[f"opt_{i}"])
        else:
            log("WARNING: Adam state does not match, continuing with fresh Adam moments")
        for k in self.hist:
            self.hist[k] = list(d[k])
        log(f"[{self.run_name}] Resumed from checkpoint: stage={self.stage}, "
            f"Adam it={self.adam_it}, L-BFGS-B it={self.lbfgs_it}, "
            f"l1={float(self.lambda_1.numpy()):.6f}, l2={float(self.lambda_2.numpy()):.6f}")

    # ------------------------------------------------------------------
    # Network and physics (unchanged)
    # ------------------------------------------------------------------
    def _build_network(self, layers):
        model = tf.keras.Sequential(name="pinn")

        for width in layers[1:-1]:
            model.add(
                tf.keras.layers.Dense(
                    width,
                    activation=tf.nn.tanh,
                    kernel_initializer=tf.keras.initializers.GlorotNormal(),
                    bias_initializer="zeros",
                    dtype=self.dtype,
                )
            )

        model.add(
            tf.keras.layers.Dense(
                layers[-1],
                activation=None,
                kernel_initializer=tf.keras.initializers.GlorotNormal(),
                bias_initializer="zeros",
                dtype=self.dtype,
            )
        )
        return model

    @property
    def train_vars(self):
        return self.net.trainable_variables + [self.lambda_1, self.lambda_2]

    def neural_net(self, X):
        # Same normalization as TF1 code
        H = 2.0 * (X - self.lb) / (self.ub - self.lb) - 1.0
        return self.net(H)

    @staticmethod
    def _safe_grad(tape, y, x):
        g = tape.gradient(y, x)
        if g is None:
            return tf.zeros_like(x)
        return g

    def net_NS(self, x, y, t):
        x = tf.convert_to_tensor(x, dtype=self.dtype)
        y = tf.convert_to_tensor(y, dtype=self.dtype)
        t = tf.convert_to_tensor(t, dtype=self.dtype)

        with tf.GradientTape(persistent=True) as tape3:
            tape3.watch([x, y, t])

            with tf.GradientTape(persistent=True) as tape2:
                tape2.watch([x, y, t])

                with tf.GradientTape(persistent=True) as tape1:
                    tape1.watch([x, y, t])

                    X = tf.concat([x, y, t], axis=1)
                    psi_and_p = self.neural_net(X)
                    psi = psi_and_p[:, 0:1]
                    p = psi_and_p[:, 1:2]

                u = self._safe_grad(tape1, psi, y)
                v = -self._safe_grad(tape1, psi, x)

                p_x = self._safe_grad(tape1, p, x)
                p_y = self._safe_grad(tape1, p, y)

            u_t = self._safe_grad(tape2, u, t)
            u_x = self._safe_grad(tape2, u, x)
            u_y = self._safe_grad(tape2, u, y)

            v_t = self._safe_grad(tape2, v, t)
            v_x = self._safe_grad(tape2, v, x)
            v_y = self._safe_grad(tape2, v, y)

        u_xx = self._safe_grad(tape3, u_x, x)
        u_yy = self._safe_grad(tape3, u_y, y)

        v_xx = self._safe_grad(tape3, v_x, x)
        v_yy = self._safe_grad(tape3, v_y, y)

        del tape1, tape2, tape3

        f_u = (
            u_t
            + self.lambda_1 * (u * u_x + v * u_y)
            + p_x
            - self.lambda_2 * (u_xx + u_yy)
        )

        f_v = (
            v_t
            + self.lambda_1 * (u * v_x + v * v_y)
            + p_y
            - self.lambda_2 * (v_xx + v_yy)
        )

        return u, v, p, f_u, f_v

    def loss(self, x, y, t, u, v):
        u_pred, v_pred, _, f_u_pred, f_v_pred = self.net_NS(x, y, t)

        loss_value = (
            tf.reduce_sum(tf.square(u - u_pred))
            + tf.reduce_sum(tf.square(v - v_pred))
            + tf.reduce_sum(tf.square(f_u_pred))
            + tf.reduce_sum(tf.square(f_v_pred))
        )
        return loss_value

    # ------------------------------------------------------------------
    # Adam (resumable)
    # ------------------------------------------------------------------
    @tf.function
    def train_step_adam(self):
        with tf.GradientTape() as tape:
            loss_value = self.loss(self.x, self.y, self.t, self.u, self.v)

        grads = tape.gradient(loss_value, self.train_vars)
        grads = [
            tf.zeros_like(v) if g is None else g
            for g, v in zip(grads, self.train_vars)
        ]
        self.optimizer_adam.apply_gradients(zip(grads, self.train_vars))
        return loss_value

    def train_adam(self, nIter):
        start_time = time.time()

        while self.adam_it < nIter:
            it = self.adam_it
            loss_value = self.train_step_adam()

            l = float(loss_value.numpy())
            l1 = float(self.lambda_1.numpy())
            l2 = float(self.lambda_2.numpy())
            self.hist["adam_loss"].append(l)
            self.hist["adam_l1"].append(l1)
            self.hist["adam_l2"].append(l2)
            self.adam_it += 1

            if it % ADAM_PRINT_EVERY == 0:
                elapsed = time.time() - start_time
                log("It: %d, Loss: %.3e, l1: %.6f, l2: %.6f, Time: %.2f"
                    % (it, l, l1, l2, elapsed))
                start_time = time.time()

            if self.adam_it % ADAM_SAVE_EVERY == 0:
                self.save_checkpoint()

    # ------------------------------------------------------------------
    # L-BFGS-B (resumable)
    # ------------------------------------------------------------------
    def _pack_weights(self):
        return np.concatenate(
            [tf.reshape(v, [-1]).numpy() for v in self.train_vars], axis=0
        ).astype(np.float64)

    def _unpack_weights(self, flat_weights):
        idx = 0
        for var in self.train_vars:
            shape = var.shape
            size = int(np.prod(shape))
            new_val = flat_weights[idx: idx + size].reshape(shape)
            var.assign(tf.convert_to_tensor(new_val, dtype=self.dtype))
            idx += size

    def _loss_and_grad_lbfgs(self, flat_weights):
        self.lbfgs_nfev += 1
        self._unpack_weights(flat_weights)

        with tf.GradientTape() as tape:
            loss_value = self.loss(self.x, self.y, self.t, self.u, self.v)

        grads = tape.gradient(loss_value, self.train_vars)
        grads = [
            tf.zeros_like(v) if g is None else g
            for g, v in zip(grads, self.train_vars)
        ]

        grad_flat = np.concatenate(
            [tf.reshape(g, [-1]).numpy() for g in grads], axis=0
        ).astype(np.float64)

        return float(loss_value.numpy()), grad_flat

    def _callback_lbfgs(self, flat_weights):
        accepted = np.array(flat_weights, copy=True)
        self._unpack_weights(accepted)
        loss_value = float(self.loss(self.x, self.y, self.t, self.u, self.v).numpy())
        l1 = float(self.lambda_1.numpy())
        l2 = float(self.lambda_2.numpy())
        # update history, counter and saved point together
        self.hist["lbfgs_loss"].append(loss_value)
        self.hist["lbfgs_l1"].append(l1)
        self.hist["lbfgs_l2"].append(l2)
        self.lbfgs_it += 1
        self._last_accepted = accepted

        log("L-BFGS-B Iter: %d/%d, Loss: %.3e, l1: %.6f, l2: %.6f"
            % (self.lbfgs_it, self.lbfgs_maxiter, loss_value, l1, l2))

        if self.lbfgs_it % LBFGS_SAVE_EVERY == 0:
            self.save_checkpoint(weights=self._last_accepted)

    def train_lbfgs(self, maxiter=50000, maxfun=100000,
                    maxcor=50, maxls=50, ftol=1.0 * np.finfo(float).eps):
        self.lbfgs_maxiter = maxiter
        remaining_it = maxiter - self.lbfgs_it
        remaining_fun = maxfun - self.lbfgs_nfev
        if remaining_it <= 0 or remaining_fun <= 0:
            return None
        if self.lbfgs_it > 0:
            # scipy cannot restore its internal curvature memory; it restarts
            # from the saved weights and rebuilds it within ~maxcor iterations
            log(f"Continuing L-BFGS-B from iteration {self.lbfgs_it} "
                f"({remaining_it} iterations left)")

        self._last_accepted = self._pack_weights()
        result = minimize(
            fun=self._loss_and_grad_lbfgs,
            x0=self._last_accepted,
            jac=True,
            method="L-BFGS-B",
            callback=self._callback_lbfgs,
            options={"maxiter": remaining_it, "maxfun": remaining_fun,
                     "maxcor": maxcor, "maxls": maxls, "ftol": ftol},
        )
        self._unpack_weights(result.x)
        log("\nL-BFGS-B finished: " + str(result.message))
        return result

    # ------------------------------------------------------------------
    # Full training with stages
    # ------------------------------------------------------------------
    def train(self, adam_iters=200000, use_lbfgs=True, lbfgs_options=None):
        if self.stage == "done":
            log(f"[{self.run_name}] already finished, skipping training")
            return None
        try:
            if self.stage == "adam":
                self.train_adam(adam_iters)
                self.stage = "lbfgs" if use_lbfgs else "done"
                self.save_checkpoint()

            if self.stage == "lbfgs":
                if lbfgs_options is None:
                    lbfgs_options = {}
                self.train_lbfgs(**lbfgs_options)
                self.stage = "done"
                self.save_checkpoint()

        except KeyboardInterrupt:
            signal.signal(signal.SIGINT, signal.SIG_IGN)    # protect the save
            # Save the last *accepted* point (L-BFGS may be inside a line search)
            w = getattr(self, "_last_accepted", None) if self.stage == "lbfgs" else None
            self.save_checkpoint(weights=w)
            log(f"\n[{self.run_name}] Stopped. Checkpoint saved at stage={self.stage}, "
                f"Adam it={self.adam_it}, L-BFGS-B it={self.lbfgs_it}. "
                f"Run the script again to continue.")
            sys.exit(0)

    def predict(self, x_star, y_star, t_star):
        x_star = tf.convert_to_tensor(x_star, dtype=self.dtype)
        y_star = tf.convert_to_tensor(y_star, dtype=self.dtype)
        t_star = tf.convert_to_tensor(t_star, dtype=self.dtype)

        u_star, v_star, p_star, _, _ = self.net_NS(x_star, y_star, t_star)

        return u_star.numpy(), v_star.numpy(), p_star.numpy()


# ----------------------------------------------------------------------
# Plot helpers (unchanged, plot_loss now uses every iteration)
# ----------------------------------------------------------------------
def plot_triplet(fig, axs, X, Y, pred, exact, label):
    """Predicted / Exact / Absolute error for one field."""
    err = np.abs(pred - exact)
    for ax, data, title in zip(
        axs,
        [pred, exact, err],
        [f"Predicted {label}", f"Exact {label}", f"Absolute error {label}"],
    ):
        h = ax.pcolormesh(X, Y, data, shading="auto", cmap="viridis")
        ax.set_title(title)
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_aspect("equal")
        fig.colorbar(h, ax=ax)


def plot_velocity(X, Y, U_pred, U_exact, V_pred, V_exact, fname="velocity_comparison.png"):
    fig, axes = plt.subplots(2, 3, figsize=(18, 9))
    plot_triplet(fig, axes[0], X, Y, U_pred, U_exact, "u")
    plot_triplet(fig, axes[1], X, Y, V_pred, V_exact, "v")
    plt.tight_layout()
    plt.savefig(fname, dpi=200)
    plt.close(fig)


def plot_pressure(X, Y, P_pred, P_exact, fname="pressure_comparison.png"):
    # Pressure is only defined up to an additive constant, so remove the mean offset
    P_pred_shifted = P_pred - (np.nanmean(P_pred) - np.nanmean(P_exact))
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    plot_triplet(fig, axes, X, Y, P_pred_shifted, P_exact, "pressure")
    plt.tight_layout()
    plt.savefig(fname, dpi=200)
    plt.close(fig)


def plot_loss(loss_adam, loss_lbfgs, adam_every=1, fname="loss_history.png"):
    it_adam = adam_every * np.arange(len(loss_adam))
    if len(loss_adam):
        it_lbfgs = it_adam[-1] + np.arange(1, len(loss_lbfgs) + 1)
    else:
        it_lbfgs = np.arange(len(loss_lbfgs))

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.semilogy(it_adam, loss_adam, label="Adam")
    if len(loss_lbfgs) > 0:
        ax.semilogy(it_lbfgs, loss_lbfgs, label="L-BFGS-B")
    ax.set_xlabel("Iteration")
    ax.set_ylabel("Loss")
    ax.set_title("Training loss")
    ax.grid(True, which="both", alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.savefig(fname, dpi=200)
    plt.close(fig)


def plot_lambdas(h, fname="lambda_history.png"):
    l1 = np.concatenate([h["adam_l1"], h["lbfgs_l1"]])
    l2 = np.concatenate([h["adam_l2"], h["lbfgs_l2"]])
    n_adam = len(h["adam_l1"])
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, val, true, name in [(axes[0], l1, 1.0, r"$\lambda_1$"),
                                (axes[1], l2, LAMBDA_2_TRUE, r"$\lambda_2$")]:
        ax.plot(val, label=name)
        ax.axhline(true, color="k", ls="--", lw=1, label="true value")
        if len(h["lbfgs_l1"]):
            ax.axvline(n_adam, color="gray", ls=":", lw=1, label="Adam -> L-BFGS-B")
        ax.set_xlabel("Iteration")
        ax.set_title(name)
        ax.grid(alpha=0.3)
        ax.legend()
    plt.tight_layout()
    plt.savefig(fname, dpi=200)
    plt.close(fig)


# ----------------------------------------------------------------------
# --status: show progress from the checkpoint files (safe while running)
# ----------------------------------------------------------------------
def show_status():
    for run in ["clean", "noisy"]:
        f = os.path.join(CKPT_DIR, f"state_{run}.npz")
        if not os.path.exists(f):
            print(f"[{run}] not started")
            continue
        d = np.load(f)
        h = {k: d[k] for k in ["adam_loss", "adam_l1", "adam_l2",
                               "lbfgs_loss", "lbfgs_l1", "lbfgs_l2"]}
        last = "lbfgs" if len(h["lbfgs_loss"]) else "adam"
        print(f"[{run}] stage={str(d['stage'])}, Adam it={int(d['adam_it'])}, "
              f"L-BFGS-B it={int(d['lbfgs_it'])}")
        if len(h[last + "_loss"]):
            l1, l2 = h[last + "_l1"][-1], h[last + "_l2"][-1]
            print(f"        last loss={h[last + '_loss'][-1]:.3e}, l1={l1:.6f} "
                  f"(err {abs(l1 - 1) * 100:.3f}%), l2={l2:.6f} "
                  f"(err {abs(l2 - LAMBDA_2_TRUE) / LAMBDA_2_TRUE * 100:.3f}%)")
            plot_loss(h["adam_loss"], h["lbfgs_loss"], fname=f"progress_loss_{run}.png")
            plot_lambdas(h, fname=f"progress_lambda_{run}.png")
            print(f"        plots: progress_loss_{run}.png, progress_lambda_{run}.png")


# ----------------------------------------------------------------------
# Main
# ----------------------------------------------------------------------
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--status", action="store_true",
                        help="print progress and save loss/lambda plots, then exit")
    parser.add_argument("--adam", type=int, default=300000)
    parser.add_argument("--lbfgs", type=int, default=100000)
    parser.add_argument("--adam-noisy", type=int, default=200000)
    parser.add_argument("--skip-noisy", action="store_true")
    # parse_known_args: ignore the extra "-f kernel.json" argument that
    # Jupyter passes, so the script also runs inside a notebook cell
    args, _ = parser.parse_known_args()

    os.makedirs(CKPT_DIR, exist_ok=True)

    if args.status:
        show_status()
        sys.exit(0)

    log("\n========== start / resume " + time.strftime("%Y-%m-%d %H:%M:%S") + " ==========")

    N_train = 5000

    layers = [3, 20, 20, 20, 20, 20, 20, 20, 20, 2]

    # Load Data
    data = scipy.io.loadmat(DATA_FILE)

    U_star = data['U_star']  # N x 2 x T
    P_star = data['p_star']  # N x T
    t_star = data['t_star']  # T x 1
    X_star = data['X_star']  # N x 2

    N = X_star.shape[0]
    T = t_star.shape[0]

    # Rearrange Data
    XX = np.tile(X_star[:, 0:1], (1, T))  # N x T
    YY = np.tile(X_star[:, 1:2], (1, T))  # N x T
    TT = np.tile(t_star, (1, N)).T        # N x T

    UU = U_star[:, 0, :]  # N x T
    VV = U_star[:, 1, :]  # N x T
    PP = P_star           # N x T

    x = XX.flatten()[:, None]  # NT x 1
    y = YY.flatten()[:, None]  # NT x 1
    t = TT.flatten()[:, None]  # NT x 1

    u = UU.flatten()[:, None]  # NT x 1
    v = VV.flatten()[:, None]  # NT x 1
    p = PP.flatten()[:, None]  # NT x 1

    # Training points and noise are saved once, so a resumed run uses
    # exactly the same data (same numbers as the original script)
    data_file = os.path.join(CKPT_DIR, "train_data.npz")
    if os.path.exists(data_file):
        d = np.load(data_file)
        idx, noise_u, noise_v = d["idx"], d["noise_u"], d["noise_v"]
    else:
        idx = np.random.choice(N * T, N_train, replace=False)
        noise_u = np.random.randn(N_train, 1)
        noise_v = np.random.randn(N_train, 1)
        np.savez(data_file, idx=idx, noise_u=noise_u, noise_v=noise_v)

    x_train = x[idx, :]
    y_train = y[idx, :]
    t_train = t[idx, :]
    u_train = u[idx, :]
    v_train = v[idx, :]

    # Test Data
    snap = np.array([100])
    x_star = X_star[:, 0:1]
    y_star = X_star[:, 1:2]
    t_star = TT[:, snap]

    u_star = U_star[:, 0, snap]
    v_star = U_star[:, 1, snap]
    p_star = P_star[:, snap]

    # ------------------------------------------------------------------
    # Noiseless run
    # ------------------------------------------------------------------
    results_file = os.path.join(CKPT_DIR, "results_clean.npz")
    if os.path.exists(results_file):
        r = np.load(results_file)
        u_pred, v_pred, p_pred = r["u_pred"], r["v_pred"], r["p_pred"]
        lambda_1_value, lambda_2_value = float(r["l1"]), float(r["l2"])
        loss_adam_hist, loss_lbfgs_hist = r["loss_adam"], r["loss_lbfgs"]
        log("[clean] results loaded from " + results_file)
    else:
        model = PhysicsInformedNN(x_train, y_train, t_train, u_train, v_train, layers,
                                  dtype=tf.float64, run_name="clean")
        model.train(
            adam_iters=args.adam,
            use_lbfgs=True,
            lbfgs_options={"maxiter": args.lbfgs, "maxfun": 3 * args.lbfgs}
        )

        loss_adam_hist = np.array(model.hist["adam_loss"])
        loss_lbfgs_hist = np.array(model.hist["lbfgs_loss"])

        # Prediction
        u_pred, v_pred, p_pred = model.predict(x_star, y_star, t_star)
        lambda_1_value = float(model.lambda_1.numpy())
        lambda_2_value = float(model.lambda_2.numpy())
        np.savez(results_file, u_pred=u_pred, v_pred=v_pred, p_pred=p_pred,
                 l1=lambda_1_value, l2=lambda_2_value,
                 loss_adam=loss_adam_hist, loss_lbfgs=loss_lbfgs_hist)

    # Error
    error_u = np.linalg.norm(u_star - u_pred, 2) / np.linalg.norm(u_star, 2)
    error_v = np.linalg.norm(v_star - v_pred, 2) / np.linalg.norm(v_star, 2)
    error_p = np.linalg.norm(p_star - p_pred, 2) / np.linalg.norm(p_star, 2)

    error_lambda_1 = np.abs(lambda_1_value - 1.0) * 100
    error_lambda_2 = np.abs(lambda_2_value - LAMBDA_2_TRUE) / LAMBDA_2_TRUE * 100

    log('Error u: %e' % (error_u))
    log('Error v: %e' % (error_v))
    log('Error p: %e' % (error_p))
    log('Error l1: %.5f%%' % (error_lambda_1))
    log('Error l2: %.5f%%' % (error_lambda_2))

    # Predict for plotting
    lb = X_star.min(0)
    ub = X_star.max(0)
    nn = 200
    xg = np.linspace(lb[0], ub[0], nn)
    yg = np.linspace(lb[1], ub[1], nn)
    X, Y = np.meshgrid(xg, yg)

    def grid_complete(points, values, X, Y):
        grid_cubic = griddata(points, values, (X, Y), method='cubic')
        grid_nearest = griddata(points, values, (X, Y), method='nearest')
        return np.where(np.isnan(grid_cubic), grid_nearest, grid_cubic)

    UU_star = grid_complete(X_star, u_pred.flatten(), X, Y)
    VV_star = grid_complete(X_star, v_pred.flatten(), X, Y)
    PP_star = grid_complete(X_star, p_pred.flatten(), X, Y)
    U_exact = grid_complete(X_star, u_star.flatten(), X, Y)
    V_exact = grid_complete(X_star, v_star.flatten(), X, Y)
    P_exact = grid_complete(X_star, p_star.flatten(), X, Y)

    # Plots of the noiseless run are written now, so they exist even if
    # the noisy run is stopped
    plot_velocity(X, Y, UU_star, U_exact, VV_star, V_exact)
    plot_pressure(X, Y, PP_star, P_exact)
    plot_loss(loss_adam_hist, loss_lbfgs_hist)

    # ------------------------------------------------------------------
    # Noisy run
    # ------------------------------------------------------------------
    if not args.skip_noisy:
        noise = 0.01
        u_train_noisy = u_train + noise * np.std(u_train) * noise_u
        v_train_noisy = v_train + noise * np.std(v_train) * noise_v

        model = PhysicsInformedNN(x_train, y_train, t_train, u_train_noisy, v_train_noisy,
                                  layers, dtype=tf.float64, run_name="noisy")
        model.train(args.adam_noisy)

        lambda_1_value_noisy = model.lambda_1.numpy()
        lambda_2_value_noisy = model.lambda_2.numpy()

        error_lambda_1_noisy = np.abs(lambda_1_value_noisy - 1.0) * 100
        error_lambda_2_noisy = np.abs(lambda_2_value_noisy - LAMBDA_2_TRUE) / LAMBDA_2_TRUE * 100

        log('Error l1 (noisy): %.5f%%' % (error_lambda_1_noisy))
        log('Error l2 (noisy): %.5f%%' % (error_lambda_2_noisy))

    log("========== all done " + time.strftime("%Y-%m-%d %H:%M:%S") + " ==========")

2026-10-22 18:01:07.818167: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.



========== start / resume 2026-10-22 18:01:14 ==========
[clean] results loaded from checkpoints_re500/results_clean.npz
Error u: 6.138371e-02
Error v: 1.600282e-01
Error p: 1.409552e+01
Error l1: 3.54725%
Error l2: 177.68847%


I0000 00:00:1792684881.105958  250724 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2015 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


[noisy] Resumed from checkpoint: stage=lbfgs, Adam it=200000, L-BFGS-B it=42800, l1=0.970934, l2=0.004467
Continuing L-BFGS-B from iteration 42800 (7200 iterations left)
L-BFGS-B Iter: 42801/50000, Loss: 4.146e+01, l1: 0.970934, l2: 0.004467
L-BFGS-B Iter: 42802/50000, Loss: 4.146e+01, l1: 0.970934, l2: 0.004467
L-BFGS-B Iter: 42803/50000, Loss: 4.146e+01, l1: 0.970934, l2: 0.004467
L-BFGS-B Iter: 42804/50000, Loss: 4.146e+01, l1: 0.970934, l2: 0.004467
L-BFGS-B Iter: 42805/50000, Loss: 4.146e+01, l1: 0.970934, l2: 0.004467
L-BFGS-B Iter: 42806/50000, Loss: 4.146e+01, l1: 0.970934, l2: 0.004466
L-BFGS-B Iter: 42807/50000, Loss: 4.146e+01, l1: 0.970934, l2: 0.004465
L-BFGS-B Iter: 42808/50000, Loss: 4.146e+01, l1: 0.970934, l2: 0.004465
L-BFGS-B Iter: 42809/50000, Loss: 4.146e+01, l1: 0.970934, l2: 0.004465
L-BFGS-B Iter: 42810/50000, Loss: 4.146e+01, l1: 0.970935, l2: 0.004465
L-BFGS-B Iter: 42811/50000, Loss: 4.146e+01, l1: 0.970935, l2: 0.004464
L-BFGS-B Iter: 42812/50000, Loss: 4.14

In [2]:
import numpy as np, scipy.io as s

d = s.loadmat('cylinder_Re500_mid.mat')
X = d['X_star']; t = d['t_star'].ravel()
print('N', X.shape[0], ' x', X[:,0].min(), X[:,0].max(), ' y', X[:,1].min(), X[:,1].max())
print('T', len(t), ' t', t.min(), t.max(), ' t[100] =', t[100])

r = np.load('checkpoints_re500/results_clean.npz')
p = d['p_star'][:, [100]]; q = r['p_pred']
print('e_p without shift [%]', 100*np.linalg.norm(p-q)/np.linalg.norm(p))
q = q - q.mean() + p.mean()
print('e_p with shift [%]',    100*np.linalg.norm(p-q)/np.linalg.norm(p))

!grep "L-BFGS-B finished\|Error" checkpoints_re500/train_log.txt

N 8934  x 1.00091052 7.9997468  y -1.99929631 1.99885583
T 501  t 0.0 100.0  t[100] = 20.0
e_p without shift [%] 1409.5517043866885
e_p with shift [%] 17.368406701142053
L-BFGS-B finished: STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT
Error u: 6.138371e-02
Error v: 1.600282e-01
Error p: 1.409552e+01
Error l1: 3.54725%
Error l2: 177.68847%
Error u: 6.138371e-02
Error v: 1.600282e-01
Error p: 1.409552e+01
Error l1: 3.54725%
Error l2: 177.68847%
L-BFGS-B finished: STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT
Error l1 (noisy): 2.86420%
Error l2 (noisy): 122.45742%
